# Week 8 Exercise: Multi-LLM Ensemble with Agreement Check

## Tweaks to the Original "Price is Right" Flow

**All LLM calls go through OpenRouter** — no direct OpenAI/Anthropic keys needed, just `OPENROUTER_API_KEY`.

**Tweak 1 — Two Frontier Agents via OpenRouter (GPT + DeepSeek)**  
The original ensemble uses GPT-5.1 + RAG as the single frontier model called directly via OpenAI. We replace it with two frontier agents, both routed through OpenRouter: one calling GPT, one calling DeepSeek. Two different LLMs collaborate on pricing via RAG.

**Tweak 2 — Agent Agreement / Confidence Check**  
Before returning the ensemble price, we measure how much the agents agree. If predictions are spread far apart, we flag low confidence.

**New Ensemble Weights:**  
- GPT Frontier via OpenRouter (RAG): 0.4  
- DeepSeek Frontier via OpenRouter (RAG): 0.4  
- Specialist (fine-tuned Llama on Modal): 0.1  
- Neural Network: 0.1

## Setup & Imports

In [ ]:
import os
import re
import logging
from typing import List, Dict
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv(override=True)

# Set up logging so we can see agent activity
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# Connect to the Chroma vector database (must already be populated from day2)
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")

print(f"Chroma collection loaded with {collection.count()} products")

## Tweak 1: Two Frontier Agents — Both via OpenRouter

We define a reusable `OpenRouterFrontierAgent` base that takes any model name.  
Then we create two instances: one for **GPT** and one for **DeepSeek**.  
Both use the same RAG logic (Chroma similarity search), but different LLM brains.

In [ ]:
from agents.agent import Agent


class OpenRouterFrontierAgent(Agent):
    """
    A Frontier Agent that calls any LLM via OpenRouter for price estimation with RAG.
    Pass in the model name to choose which LLM to use.
    """

    color = Agent.CYAN

    def __init__(self, collection, model_name="deepseek/deepseek-chat", agent_label="OpenRouter Frontier"):
        self.name = agent_label
        self.model_name = model_name
        self.log(f"Initializing {agent_label} ({model_name}) via OpenRouter")
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ.get("OPENROUTER_API_KEY"),
        )
        self.collection = collection
        self.encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.log(f"{agent_label} is ready")

    def make_context(self, similars: List[str], prices: List[float]) -> str:
        message = "To provide some context, here are some similar items for reference.\n\n"
        for similar, price in zip(similars, prices):
            message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
        return message

    def messages_for(self, description: str, similars: List[str], prices: List[float]) -> List[Dict[str, str]]:
        message = f"Estimate the price of this product. Respond with the price only, no explanation.\n\n{description}\n\n"
        message += self.make_context(similars, prices)
        return [{"role": "user", "content": message}]

    def find_similars(self, description: str):
        self.log("Performing RAG search in Chroma for 5 similar products")
        vector = self.encoder.encode([description])
        results = self.collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)
        documents = results["documents"][0][:]
        prices = [m["price"] for m in results["metadatas"][0][:]]
        self.log("Found similar products")
        return documents, prices

    def get_price(self, s) -> float:
        s = s.replace("$", "").replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", s)
        return float(match.group()) if match else 0.0

    def price(self, description: str) -> float:
        documents, prices = self.find_similars(description)
        self.log(f"Calling {self.model_name} via OpenRouter with RAG context")
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=self.messages_for(description, documents, prices),
            seed=42,
        )
        reply = response.choices[0].message.content
        result = self.get_price(reply)
        self.log(f"{self.name} completed - predicting ${result:.2f}")
        return result

In [ ]:
# Quick test - both frontier agents via OpenRouter
test_description = "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"

gpt_agent = OpenRouterFrontierAgent(collection, model_name="openai/gpt-4o-mini", agent_label="GPT Frontier")
deepseek_agent = OpenRouterFrontierAgent(collection, model_name="deepseek/deepseek-chat", agent_label="DeepSeek Frontier")

print("GPT estimate:", gpt_agent.price(test_description))
print("DeepSeek estimate:", deepseek_agent.price(test_description))

## Tweak 2: Enhanced Ensemble Agent with Agreement Check

Key changes from the original:
1. **Both frontier agents use OpenRouter** — GPT and DeepSeek, no direct API keys needed
2. **Agreement check** logs confidence level based on how far apart predictions are
3. **Re-weighted** to split frontier weight equally between the two models

In [ ]:
from agents.specialist_agent import SpecialistAgent
from agents.neural_network_agent import NeuralNetworkAgent
from agents.preprocessor import Preprocessor


class EnhancedEnsembleAgent(Agent):
    """
    An enhanced Ensemble Agent that uses TWO frontier models (GPT + DeepSeek),
    both accessed via OpenRouter, alongside the fine-tuned Specialist and Neural Network.
    Includes an agreement check to gauge prediction confidence.
    """

    name = "Enhanced Ensemble Agent"
    color = Agent.YELLOW

    def __init__(self, collection):
        self.log("Initializing Enhanced Ensemble Agent")
        self.specialist = SpecialistAgent()
        self.frontier_gpt = OpenRouterFrontierAgent(
            collection, model_name="openai/gpt-4o-mini", agent_label="GPT Frontier"
        )
        self.frontier_deepseek = OpenRouterFrontierAgent(
            collection, model_name="deepseek/deepseek-chat", agent_label="DeepSeek Frontier"
        )
        self.neural_network = NeuralNetworkAgent()
        self.preprocessor = Preprocessor()
        self.log("Enhanced Ensemble Agent is ready with 4 sub-agents (all LLMs via OpenRouter)")

    def check_agreement(self, prices: dict) -> str:
        """
        Measure how much the agents agree on the price.
        Returns HIGH / MEDIUM / LOW confidence based on spread between predictions.
        """
        values = list(prices.values())
        spread = max(values) - min(values)
        avg = sum(values) / len(values)
        spread_pct = (spread / avg * 100) if avg > 0 else 0

        if spread_pct < 30:
            confidence = "HIGH"
        elif spread_pct < 60:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"

        self.log(f"Agreement Check: {confidence} confidence "
                 f"(spread: ${spread:.2f}, {spread_pct:.0f}% of avg ${avg:.2f})")
        for name, val in prices.items():
            self.log(f"  - {name}: ${val:.2f}")
        return confidence

    def price(self, description: str) -> float:
        """
        Run the multi-LLM ensemble (all via OpenRouter):
        GPT Frontier (0.4) + DeepSeek Frontier (0.4) + Specialist (0.1) + Neural Network (0.1)
        """
        self.log("Running Enhanced Ensemble - preprocessing text")
        rewrite = self.preprocessor.preprocess(description)
        self.log(f"Pre-processed text using {self.preprocessor.model_name}")

        # Get predictions from all 4 agents
        specialist = self.specialist.price(rewrite)
        frontier_gpt = self.frontier_gpt.price(rewrite)
        frontier_deepseek = self.frontier_deepseek.price(rewrite)
        neural_network = self.neural_network.price(rewrite)

        # Agreement check
        self.check_agreement({
            "GPT Frontier (RAG)": frontier_gpt,
            "DeepSeek Frontier (RAG)": frontier_deepseek,
            "Specialist (Llama)": specialist,
            "Neural Network": neural_network,
        })

        # Weighted combination
        combined = (frontier_gpt * 0.4
                    + frontier_deepseek * 0.4
                    + specialist * 0.1
                    + neural_network * 0.1)

        self.log(f"Enhanced Ensemble complete - returning ${combined:.2f}")
        return combined

## Test the Enhanced Ensemble

Let's initialize and test with a few product descriptions to see all 4 agents collaborate and the agreement check in action.

In [ ]:
enhanced_ensemble = EnhancedEnsembleAgent(collection)

In [ ]:
# Test 1: A mic - should be in the $50-150 range
enhanced_ensemble.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
# Test 2: A more expensive item
enhanced_ensemble.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

## Evaluate: Compare Original vs Enhanced Ensemble

Run both ensembles on the test set to see if adding DeepSeek as a second frontier model improves accuracy.

In [ ]:
from agents.items import Item
from agents.evaluator import evaluate

# Load the test dataset
dataset = "ed-donner/items_full"
train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(test):,} test items")

In [ ]:
# Evaluate the enhanced ensemble (4 agents, both frontiers via OpenRouter + agreement check)
def enhanced_ensemble_fn(item):
    return enhanced_ensemble.price(item.summary)

evaluate(enhanced_ensemble_fn, test)

## Summary

**What changed from the original:**

1. **All LLM calls via OpenRouter** — Instead of requiring separate API keys for OpenAI, everything routes through a single `OPENROUTER_API_KEY`. The `OpenRouterFrontierAgent` class is reusable with any model name.

2. **Two Frontier Agents** — GPT (`openai/gpt-4o-mini`) and DeepSeek (`deepseek/deepseek-chat`) both do RAG-based price estimation independently. Two different LLMs reasoning about the same product with the same context reduces the chance of any single model's blind spots affecting the final price.

3. **Agreement Check** — Before returning the ensemble price, we calculate the spread across all 4 agent predictions. HIGH (<30% spread), MEDIUM (30-60%), or LOW (>60%) confidence is logged. This flags products where the ensemble might be unreliable.

4. **Re-weighted Ensemble** — Original: GPT 0.8 + Specialist 0.1 + NN 0.1. New: GPT 0.4 + DeepSeek 0.4 + Specialist 0.1 + NN 0.1. The frontier weight is split equally between two models for a more balanced multi-LLM approach.